# Notebook 06 — Front Detection Failure Audit

**Objective:** audit exact-count, undercount, and overcount patterns from Notebook 05 on the continuous raw Front video. Count mismatches are diagnostic signals—not official false positives or false negatives—because per-frame ground-truth boxes are unavailable.

Permitted terminology: **EXACT-COUNT FRAME**, **UNDERCOUNT FRAME**, **OVERCOUNT FRAME**, **SUSPECTED MISS**, and **SUSPECTED DUPLICATE DETECTION**. Automatic image measurements remain diagnostics and are not ground-truth failure labels.

This notebook does not train, track, modify the model/dataset, classify behavior, or perform Notebook 07.

## 1. CONFIG, source paths, and runtime metadata

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, platform, subprocess, sys, time
import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
STARTED_AT = datetime.now(timezone.utc).isoformat()
GIT_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, capture_output=True, text=True, check=True).stdout.strip()
CONDA_ENV = os.environ.get('CONDA_DEFAULT_ENV', '')
CUDA_AVAILABLE = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else 'NOT_AVAILABLE'
# Must match the accepted Notebook 05 operational threshold. Change only this value to audit another completed threshold run.
DETECTION_CONF = 0.68
EXPECTED_FISH_COUNT = 1
CONF_MARGIN = 0.10
EDGE_MARGIN_FRACTION = 0.05
HIGH_OVERLAP_IOU = 0.50
MAX_REVIEW_FRAMES = 60
FLOAT_TOLERANCE = 1e-6
if not (0.0 < DETECTION_CONF <= 1.0): raise ValueError(f'DETECTION_CONF must be in (0, 1], got {DETECTION_CONF}')
if not (0.0 < CONF_MARGIN <= 1.0): raise ValueError(f'CONF_MARGIN must be in (0, 1], got {CONF_MARGIN}')
CONF_TAG = f'conf{int(round(DETECTION_CONF * 100)):03d}'
FISH_COUNT_TAG = f'n{EXPECTED_FISH_COUNT}'
RUN_TAG = f'{CONF_TAG}_{FISH_COUNT_TAG}'
SOURCE_EXPERIMENT_ID = f'FRONT_VIDEO_DET_{CONF_TAG.upper()}_{FISH_COUNT_TAG.upper()}_001'
EXPERIMENT_ID = f'FRONT_DET_FAILURE_AUDIT_{CONF_TAG.upper()}_{FISH_COUNT_TAG.upper()}_001'
SOURCE_LOG_DIR = PROJECT_ROOT / 'logs' / 'detection' / SOURCE_EXPERIMENT_ID
SOURCE_SUMMARY_PATH = SOURCE_LOG_DIR / 'summary.json'
SOURCE_CONFIG_PATH = SOURCE_LOG_DIR / 'config.yaml'
SOURCE_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'front' / 'detection' / RUN_TAG
FRAME_COUNTS_PATH = SOURCE_OUTPUT_DIR / 'frame_counts.csv'
DETECTIONS_PATH = SOURCE_OUTPUT_DIR / 'detections.csv'
REVIEW_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'front' / 'detection' / 'failure_audit' / RUN_TAG
CONFIG = {'experiment_id': EXPERIMENT_ID, 'source_experiment_id': SOURCE_EXPERIMENT_ID, 'detection_conf': DETECTION_CONF, 'expected_fish_count': EXPECTED_FISH_COUNT, 'experimental_fish_count': EXPECTED_FISH_COUNT, 'conf_tag': CONF_TAG, 'fish_count_tag': FISH_COUNT_TAG, 'run_tag': RUN_TAG, 'conf_margin': CONF_MARGIN, 'edge_margin_fraction': EDGE_MARGIN_FRACTION, 'high_overlap_iou': HIGH_OVERLAP_IOU, 'max_review_frames': MAX_REVIEW_FRAMES, 'source_summary': str(SOURCE_SUMMARY_PATH.relative_to(PROJECT_ROOT)), 'frame_counts': str(FRAME_COUNTS_PATH.relative_to(PROJECT_ROOT)), 'detections': str(DETECTIONS_PATH.relative_to(PROJECT_ROOT)), 'review_output_dir': str(REVIEW_OUTPUT_DIR.relative_to(PROJECT_ROOT))}
print('CONFIG — FRONT DETECTION FAILURE AUDIT')
print(yaml.safe_dump(CONFIG, sort_keys=False))
print(f'datetime_utc: {STARTED_AT}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}; Conda env: {CONDA_ENV}')
print(f'Python: {platform.python_version()}; Torch: {torch.__version__}; Ultralytics: {ultralytics.__version__}')
print(f'CUDA available: {CUDA_AVAILABLE}; GPU: {GPU_NAME}; Git commit: {GIT_COMMIT}')

CONFIG — FRONT DETECTION FAILURE AUDIT
experiment_id: FRONT_DET_FAILURE_AUDIT_CONF068_N1_001
source_experiment_id: FRONT_VIDEO_DET_CONF068_N1_001
detection_conf: 0.68
expected_fish_count: 1
experimental_fish_count: 1
conf_tag: conf068
fish_count_tag: n1
run_tag: conf068_n1
conf_margin: 0.1
edge_margin_fraction: 0.05
high_overlap_iou: 0.5
max_review_frames: 60
source_summary: logs/detection/FRONT_VIDEO_DET_CONF068_N1_001/summary.json
frame_counts: outputs/front/detection/conf068_n1/frame_counts.csv
detections: outputs/front/detection/conf068_n1/detections.csv
review_output_dir: outputs/front/detection/failure_audit/conf068_n1

datetime_utc: 2026-08-17T08:58:45.784069+00:00
PROJECT_ROOT: /home/diy-hus/fish
Python executable: /home/diy-hus/miniconda3/envs/fish/bin/python; Conda env: fish
Python: 3.11.15; Torch: 2.13.0+cu130; Ultralytics: 8.4.120
CUDA available: True; GPU: NVIDIA GeForce RTX 3050; Git commit: 4cfac689c08d0f3f25bdee9cb8aac99d3202b9ee


## 2. Mandatory source provenance and completeness preflight

In [2]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''): digest.update(chunk)
    return digest.hexdigest()
assert SOURCE_SUMMARY_PATH.is_file(), f'FAIL preflight: missing {SOURCE_SUMMARY_PATH}'
assert SOURCE_CONFIG_PATH.is_file(), f'FAIL preflight: missing {SOURCE_CONFIG_PATH}'
assert FRAME_COUNTS_PATH.is_file(), f'FAIL preflight: missing {FRAME_COUNTS_PATH}'
assert DETECTIONS_PATH.is_file(), f'FAIL preflight: missing {DETECTIONS_PATH}'
with SOURCE_SUMMARY_PATH.open(encoding='utf-8') as handle: SOURCE_SUMMARY = json.load(handle)
with SOURCE_CONFIG_PATH.open(encoding='utf-8') as handle: SOURCE_CONFIG = yaml.safe_load(handle)
assert SOURCE_SUMMARY['experiment_id'] == SOURCE_EXPERIMENT_ID, 'FAIL preflight: source experiment ID mismatch.'
assert math.isclose(float(SOURCE_SUMMARY['detection_conf']), DETECTION_CONF, abs_tol=FLOAT_TOLERANCE), 'FAIL preflight: confidence mismatch.'
assert int(SOURCE_SUMMARY['expected_fish_count']) == EXPECTED_FISH_COUNT, 'FAIL preflight: expected fish count mismatch.'
assert math.isclose(float(SOURCE_CONFIG['detection_conf']), DETECTION_CONF, abs_tol=FLOAT_TOLERANCE), 'FAIL preflight: source config confidence mismatch.'
assert SOURCE_SUMMARY['model_sha256'] == SOURCE_CONFIG['expected_model_sha256'], 'FAIL preflight: model SHA-256 mismatch between source summary/config.'
VIDEO_PATH = PROJECT_ROOT / SOURCE_SUMMARY['video']
MODEL_PATH = PROJECT_ROOT / SOURCE_SUMMARY['model']
assert VIDEO_PATH.is_file(), f'FAIL preflight: source video missing: {VIDEO_PATH}'
assert MODEL_PATH.is_file(), f'FAIL preflight: source model missing: {MODEL_PATH}'
VIDEO_SHA256 = sha256_file(VIDEO_PATH); MODEL_SHA256 = sha256_file(MODEL_PATH)
assert VIDEO_SHA256 == SOURCE_SUMMARY['video_sha256'], 'FAIL preflight: video SHA-256 mismatch.'
assert MODEL_SHA256 == SOURCE_SUMMARY['model_sha256'], 'FAIL preflight: model SHA-256 mismatch.'
TOTAL_FRAMES = int(SOURCE_SUMMARY['frames']); VIDEO_FPS = float(SOURCE_SUMMARY['fps'])
VIDEO_WIDTH, VIDEO_HEIGHT = map(int, SOURCE_SUMMARY['resolution'].split('x'))
FRAME_COUNTS_RAW = pd.read_csv(FRAME_COUNTS_PATH)
DETECTIONS_RAW = pd.read_csv(DETECTIONS_PATH)
required_frame_columns = {'frame_index', 'time_sec', 'n_detections'}
required_detection_columns = {'frame_index', 'time_sec', 'class_id', 'confidence', 'x1', 'y1', 'x2', 'y2', 'cx', 'cy'}
assert required_frame_columns.issubset(FRAME_COUNTS_RAW.columns), f'FAIL preflight: frame columns missing: {required_frame_columns - set(FRAME_COUNTS_RAW.columns)}'
assert required_detection_columns.issubset(DETECTIONS_RAW.columns), f'FAIL preflight: detection columns missing: {required_detection_columns - set(DETECTIONS_RAW.columns)}'
assert len(FRAME_COUNTS_RAW) == TOTAL_FRAMES, f'FAIL preflight: frame table has {len(FRAME_COUNTS_RAW)}/{TOTAL_FRAMES} rows.'
assert FRAME_COUNTS_RAW['frame_index'].astype(int).tolist() == list(range(TOTAL_FRAMES)), 'FAIL preflight: frame indices are not complete and contiguous.'
counts_from_detections = DETECTIONS_RAW.groupby('frame_index').size().reindex(range(TOTAL_FRAMES), fill_value=0).to_numpy()
assert np.array_equal(counts_from_detections, FRAME_COUNTS_RAW['n_detections'].to_numpy(dtype=int)), 'FAIL preflight: per-frame counts disagree with detections.csv.'
if len(DETECTIONS_RAW): assert float(DETECTIONS_RAW['confidence'].min()) >= DETECTION_CONF - FLOAT_TOLERANCE, 'FAIL preflight: retained detection below configured confidence.'
if REVIEW_OUTPUT_DIR.exists() and any(REVIEW_OUTPUT_DIR.iterdir()): raise RuntimeError(f'FAIL preflight: preserve existing review output before rerun: {REVIEW_OUTPUT_DIR}')
print(f'Source experiment: {SOURCE_EXPERIMENT_ID}')
print(f'Model SHA-256: {MODEL_SHA256}')
print(f'Video: {VIDEO_PATH.relative_to(PROJECT_ROOT)}; SHA-256: {VIDEO_SHA256}')
print(f'Detection confidence: {DETECTION_CONF}; expected fish: {EXPECTED_FISH_COUNT}; frames: {TOTAL_FRAMES}; FPS: {VIDEO_FPS:.6f}; resolution: {VIDEO_WIDTH}x{VIDEO_HEIGHT}')
print(f'Detections loaded: {len(DETECTIONS_RAW)}')
print('PREFLIGHT_RESULT: PASS')

Source experiment: FRONT_VIDEO_DET_CONF068_N1_001
Model SHA-256: 750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738
Video: data/raw/front/4.mp4; SHA-256: 3f8587344beba8dcc6f835d1ed94aa8daadb06a369b5b96977ee902f035b5700
Detection confidence: 0.68; expected fish: 1; frames: 3431; FPS: 28.668432; resolution: 1280x960
Detections loaded: 2301
PREFLIGHT_RESULT: PASS


## 3. Frame-level count classification and error severity

In [3]:
FRAME_AUDIT = FRAME_COUNTS_RAW.copy()
FRAME_AUDIT['frame_index'] = FRAME_AUDIT['frame_index'].astype(int)
FRAME_AUDIT['count_error'] = FRAME_AUDIT['n_detections'].astype(int) - EXPECTED_FISH_COUNT
FRAME_AUDIT['absolute_count_error'] = FRAME_AUDIT['count_error'].abs()
FRAME_AUDIT['error_type'] = np.select([FRAME_AUDIT['count_error'] < 0, FRAME_AUDIT['count_error'] > 0], ['UNDERCOUNT', 'OVERCOUNT'], default='EXACT')
FRAME_AUDIT['severity'] = np.select([FRAME_AUDIT['absolute_count_error'] == 0, FRAME_AUDIT['absolute_count_error'] == 1, FRAME_AUDIT['absolute_count_error'] == 2], ['error_0', 'error_1', 'error_2'], default='error_ge3')
EXACT_FRAMES = int((FRAME_AUDIT['error_type'] == 'EXACT').sum()); UNDERCOUNT_FRAMES = int((FRAME_AUDIT['error_type'] == 'UNDERCOUNT').sum()); OVERCOUNT_FRAMES = int((FRAME_AUDIT['error_type'] == 'OVERCOUNT').sum())
EXACT_RATE = EXACT_FRAMES / TOTAL_FRAMES; UNDERCOUNT_RATE = UNDERCOUNT_FRAMES / TOTAL_FRAMES; OVERCOUNT_RATE = OVERCOUNT_FRAMES / TOTAL_FRAMES
ERROR_ABS_1_FRAMES = int((FRAME_AUDIT['absolute_count_error'] == 1).sum()); ERROR_ABS_2_FRAMES = int((FRAME_AUDIT['absolute_count_error'] == 2).sum()); ERROR_ABS_GE3_FRAMES = int((FRAME_AUDIT['absolute_count_error'] >= 3).sum())
severity_rows = []
for direction in ('EXACT', 'UNDERCOUNT', 'OVERCOUNT'):
    subset = FRAME_AUDIT[FRAME_AUDIT['error_type'] == direction]
    for severity in ('error_0', 'error_1', 'error_2', 'error_ge3'):
        frames = int((subset['severity'] == severity).sum())
        severity_rows.append({'error_type': direction, 'severity': severity, 'frames': frames, 'frame_rate_all': frames / TOTAL_FRAMES, 'rate_within_error_type': frames / len(subset) if len(subset) else 0.0})
SEVERITY_DF = pd.DataFrame(severity_rows)
display(FRAME_AUDIT[['frame_index', 'time_sec', 'n_detections', 'count_error', 'absolute_count_error', 'error_type', 'severity']].head())
display(SEVERITY_DF[SEVERITY_DF['frames'] > 0])
print(f'EXACT-COUNT frames: {EXACT_FRAMES} ({EXACT_RATE:.4f})')
print(f'UNDERCOUNT frames: {UNDERCOUNT_FRAMES} ({UNDERCOUNT_RATE:.4f})')
print(f'OVERCOUNT frames: {OVERCOUNT_FRAMES} ({OVERCOUNT_RATE:.4f})')

,frame_index,time_sec,n_detections,count_error,absolute_count_error,error_type,severity
0,0,0.000000,1,0,0,EXACT,error_0
1,1,0.034882,1,0,0,EXACT,error_0
2,2,0.069763,1,0,0,EXACT,error_0
3,3,0.104645,1,0,0,EXACT,error_0
4,4,0.139526,1,0,0,EXACT,error_0


,error_type,severity,frames,frame_rate_all,rate_within_error_type
0,EXACT,error_0,2301,0.67065,1.0
5,UNDERCOUNT,error_1,1130,0.32935,1.0


EXACT-COUNT frames: 2301 (0.6706)
UNDERCOUNT frames: 1130 (0.3294)
OVERCOUNT frames: 0 (0.0000)


## 4. Consecutive temporal error runs

In [4]:
error_runs = []
active_start = None; active_type = None
for row in FRAME_AUDIT.itertuples(index=False):
    current_type = row.error_type if row.error_type != 'EXACT' else None
    if current_type != active_type:
        if active_type is not None:
            run = FRAME_AUDIT.iloc[active_start:row.frame_index]
            error_runs.append({'start_frame': int(run['frame_index'].iloc[0]), 'end_frame': int(run['frame_index'].iloc[-1]), 'duration_frames': int(len(run)), 'duration_sec': float(len(run) / VIDEO_FPS), 'error_type': active_type, 'mean_count': float(run['n_detections'].mean()), 'minimum_count': int(run['n_detections'].min()), 'maximum_count': int(run['n_detections'].max())})
        active_start = row.frame_index if current_type is not None else None
        active_type = current_type
if active_type is not None:
    run = FRAME_AUDIT.iloc[active_start:TOTAL_FRAMES]
    error_runs.append({'start_frame': int(run['frame_index'].iloc[0]), 'end_frame': int(run['frame_index'].iloc[-1]), 'duration_frames': int(len(run)), 'duration_sec': float(len(run) / VIDEO_FPS), 'error_type': active_type, 'mean_count': float(run['n_detections'].mean()), 'minimum_count': int(run['n_detections'].min()), 'maximum_count': int(run['n_detections'].max())})
ERROR_RUNS_DF = pd.DataFrame(error_runs, columns=['start_frame', 'end_frame', 'duration_frames', 'duration_sec', 'error_type', 'mean_count', 'minimum_count', 'maximum_count'])
ERROR_RUN_COUNT = len(ERROR_RUNS_DF)
LONGEST_ERROR_RUN_FRAMES = int(ERROR_RUNS_DF['duration_frames'].max()) if ERROR_RUN_COUNT else 0
LONGEST_ERROR_RUN_SEC = float(ERROR_RUNS_DF['duration_sec'].max()) if ERROR_RUN_COUNT else 0.0
display(ERROR_RUNS_DF.sort_values('duration_frames', ascending=False).head(20))
print(f'Error runs: {ERROR_RUN_COUNT}; longest: {LONGEST_ERROR_RUN_FRAMES} frames ({LONGEST_ERROR_RUN_SEC:.3f} sec)')

,start_frame,end_frame,duration_frames,duration_sec,error_type,mean_count,minimum_count,maximum_count
14,971,2020,1050,36.625652,UNDERCOUNT,0.0,0,0
25,2600,2616,17,0.592987,UNDERCOUNT,0.0,0,0
1,239,246,8,0.279053,UNDERCOUNT,0.0,0,0
9,418,424,7,0.244171,UNDERCOUNT,0.0,0,0
11,913,916,4,0.139526,UNDERCOUNT,0.0,0,0
2,362,365,4,0.139526,UNDERCOUNT,0.0,0,0
28,2884,2886,3,0.104645,UNDERCOUNT,0.0,0,0
0,209,211,3,0.104645,UNDERCOUNT,0.0,0,0
13,925,926,2,0.069763,UNDERCOUNT,0.0,0,0
12,918,919,2,0.069763,UNDERCOUNT,0.0,0,0


Error runs: 32; longest: 1050 frames (36.626 sec)


## 5. Near-threshold and automatic bbox diagnostics

In [5]:
DETECTIONS = DETECTIONS_RAW.copy()
DETECTIONS['near_threshold'] = (DETECTIONS['confidence'] >= DETECTION_CONF - FLOAT_TOLERANCE) & (DETECTIONS['confidence'] < DETECTION_CONF + CONF_MARGIN)
DETECTIONS['bbox_width'] = DETECTIONS['x2'] - DETECTIONS['x1']; DETECTIONS['bbox_height'] = DETECTIONS['y2'] - DETECTIONS['y1']
DETECTIONS['bbox_area'] = DETECTIONS['bbox_width'] * DETECTIONS['bbox_height']
DETECTIONS['near_frame_edge'] = (DETECTIONS['x1'] <= EDGE_MARGIN_FRACTION * VIDEO_WIDTH) | (DETECTIONS['y1'] <= EDGE_MARGIN_FRACTION * VIDEO_HEIGHT) | (DETECTIONS['x2'] >= (1 - EDGE_MARGIN_FRACTION) * VIDEO_WIDTH) | (DETECTIONS['y2'] >= (1 - EDGE_MARGIN_FRACTION) * VIDEO_HEIGHT)
NEAR_THRESHOLD_DETECTIONS = int(DETECTIONS['near_threshold'].sum()); NEAR_THRESHOLD_RATE = NEAR_THRESHOLD_DETECTIONS / len(DETECTIONS) if len(DETECTIONS) else 0.0
NEAR_THRESHOLD_FRAMES = int(DETECTIONS.loc[DETECTIONS['near_threshold'], 'frame_index'].nunique())
def pairwise_diagnostics(group):
    values = group[['x1', 'y1', 'x2', 'y2', 'cx', 'cy']].to_numpy(dtype=float)
    min_center_distance = float('nan'); max_iou = 0.0
    for left in range(len(values)):
        for right in range(left + 1, len(values)):
            a, b = values[left], values[right]
            distance = math.hypot(a[4] - b[4], a[5] - b[5])
            min_center_distance = distance if math.isnan(min_center_distance) else min(min_center_distance, distance)
            ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
            intersection = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
            union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - intersection
            max_iou = max(max_iou, intersection / union if union > 0 else 0.0)
    return pd.Series({'mean_detection_confidence': float(group['confidence'].mean()), 'median_detection_confidence': float(group['confidence'].median()), 'minimum_detection_confidence': float(group['confidence'].min()), 'mean_bbox_area': float(group['bbox_area'].mean()), 'minimum_center_distance': min_center_distance, 'maximum_pairwise_iou': max_iou, 'near_threshold_detections': int(group['near_threshold'].sum()), 'near_edge_detections': int(group['near_frame_edge'].sum()), 'high_bbox_overlap': bool(max_iou >= HIGH_OVERLAP_IOU)})
if len(DETECTIONS):
    PER_FRAME_IMAGE_DIAGNOSTICS = DETECTIONS.groupby('frame_index', sort=True).apply(pairwise_diagnostics, include_groups=False).reset_index()
else:
    PER_FRAME_IMAGE_DIAGNOSTICS = pd.DataFrame(columns=['frame_index', 'mean_detection_confidence', 'median_detection_confidence', 'minimum_detection_confidence', 'mean_bbox_area', 'minimum_center_distance', 'maximum_pairwise_iou', 'near_threshold_detections', 'near_edge_detections', 'high_bbox_overlap'])
FRAME_AUDIT = FRAME_AUDIT.merge(PER_FRAME_IMAGE_DIAGNOSTICS, on='frame_index', how='left')
FRAME_AUDIT[['near_threshold_detections', 'near_edge_detections']] = FRAME_AUDIT[['near_threshold_detections', 'near_edge_detections']].fillna(0).astype(int)
FRAME_AUDIT['high_bbox_overlap'] = FRAME_AUDIT['high_bbox_overlap'].fillna(False).astype(bool)
print(f'Near-threshold range: [{DETECTION_CONF:.3f}, {DETECTION_CONF + CONF_MARGIN:.3f})')
print(f'Near-threshold detections: {NEAR_THRESHOLD_DETECTIONS}/{len(DETECTIONS)} ({NEAR_THRESHOLD_RATE:.4f}); affected frames: {NEAR_THRESHOLD_FRAMES}')

Near-threshold range: [0.680, 0.780)
Near-threshold detections: 1663/2301 (0.7227); affected frames: 1663


## 6. EXACT versus ERROR association summary

In [6]:
FRAME_AUDIT['comparison_group'] = np.where(FRAME_AUDIT['error_type'] == 'EXACT', 'EXACT', 'ERROR')
comparison_metrics = ['mean_detection_confidence', 'median_detection_confidence', 'minimum_detection_confidence', 'mean_bbox_area', 'minimum_center_distance', 'maximum_pairwise_iou', 'near_threshold_detections', 'near_edge_detections']
comparison_rows = []
for group_name, group in FRAME_AUDIT.groupby('comparison_group'):
    row = {'frame_group': group_name, 'frames': len(group)}
    for metric in comparison_metrics: row[f'mean_{metric}'] = float(group[metric].mean()) if group[metric].notna().any() else None
    row['high_bbox_overlap_frame_rate'] = float(group['high_bbox_overlap'].mean())
    comparison_rows.append(row)
EXACT_VS_ERROR_DF = pd.DataFrame(comparison_rows)
display(EXACT_VS_ERROR_DF)
print('These comparisons describe associations only; they do not establish causes or official FP/FN labels.')

,frame_group,frames,mean_mean_detection_confidence,mean_median_detection_confidence,mean_minimum_detection_confidence,mean_mean_bbox_area,mean_minimum_center_distance,mean_maximum_pairwise_iou,mean_near_threshold_detections,mean_near_edge_detections,high_bbox_overlap_frame_rate
0,ERROR,1130,NaN,NaN,NaN,NaN,None,NaN,0.000000,0.0,0.0
1,EXACT,2301,0.748629,0.748629,0.748629,15801.588651,None,0.0,0.722729,0.0,0.0


These comparisons describe associations only; they do not establish causes or official FP/FN labels.


## 7. Deterministic review-frame selection

In [7]:
review_reasons = {}
def add_review_frames(frame_indices, reason):
    for value in frame_indices:
        frame_index = int(value)
        if 0 <= frame_index < TOTAL_FRAMES: review_reasons.setdefault(frame_index, set()).add(reason)
add_review_frames(FRAME_AUDIT[FRAME_AUDIT['error_type'] == 'UNDERCOUNT'].sort_values(['count_error', 'frame_index']).head(15)['frame_index'], 'HEAVIEST_UNDERCOUNT')
add_review_frames(FRAME_AUDIT[FRAME_AUDIT['error_type'] == 'OVERCOUNT'].sort_values(['count_error', 'frame_index'], ascending=[False, True]).head(15)['frame_index'], 'HEAVIEST_OVERCOUNT')
if ERROR_RUN_COUNT:
    for run in ERROR_RUNS_DF.sort_values('duration_frames', ascending=False).head(10).itertuples(index=False):
        add_review_frames([run.start_frame, (run.start_frame + run.end_frame) // 2, run.end_frame], 'LONG_ERROR_RUN')
add_review_frames(FRAME_AUDIT.sort_values(['near_threshold_detections', 'frame_index'], ascending=[False, True]).head(15)['frame_index'], 'NEAR_THRESHOLD_CLUSTER')
exact_indices = FRAME_AUDIT.loc[FRAME_AUDIT['error_type'] == 'EXACT', 'frame_index'].to_numpy(dtype=int)
if len(exact_indices):
    exact_positions = np.linspace(0, len(exact_indices) - 1, min(10, len(exact_indices)), dtype=int)
    add_review_frames(exact_indices[exact_positions], 'EXACT_CONTROL')
if len(review_reasons) > MAX_REVIEW_FRAMES:
    ranked = sorted(review_reasons, key=lambda frame: (-FRAME_AUDIT.loc[FRAME_AUDIT.frame_index == frame, 'absolute_count_error'].iloc[0], -FRAME_AUDIT.loc[FRAME_AUDIT.frame_index == frame, 'near_threshold_detections'].iloc[0], frame))[:MAX_REVIEW_FRAMES]
    review_reasons = {frame: review_reasons[frame] for frame in ranked}
REVIEW_FRAMES = sorted(review_reasons)
assert REVIEW_FRAMES, 'FAIL: no review frames were selected.'
print(f'Review frames selected: {len(REVIEW_FRAMES)}')
for frame in REVIEW_FRAMES: print(f'- frame {frame}: {"|".join(sorted(review_reasons[frame]))}')

Review frames selected: 58
- frame 0: EXACT_CONTROL|NEAR_THRESHOLD_CLUSTER
- frame 1: NEAR_THRESHOLD_CLUSTER
- frame 2: NEAR_THRESHOLD_CLUSTER
- frame 3: NEAR_THRESHOLD_CLUSTER
- frame 4: NEAR_THRESHOLD_CLUSTER
- frame 5: NEAR_THRESHOLD_CLUSTER
- frame 6: NEAR_THRESHOLD_CLUSTER
- frame 7: NEAR_THRESHOLD_CLUSTER
- frame 8: NEAR_THRESHOLD_CLUSTER
- frame 9: NEAR_THRESHOLD_CLUSTER
- frame 10: NEAR_THRESHOLD_CLUSTER
- frame 11: NEAR_THRESHOLD_CLUSTER
- frame 12: NEAR_THRESHOLD_CLUSTER
- frame 13: NEAR_THRESHOLD_CLUSTER
- frame 14: NEAR_THRESHOLD_CLUSTER
- frame 209: HEAVIEST_UNDERCOUNT|LONG_ERROR_RUN
- frame 210: HEAVIEST_UNDERCOUNT|LONG_ERROR_RUN
- frame 211: HEAVIEST_UNDERCOUNT|LONG_ERROR_RUN
- frame 239: HEAVIEST_UNDERCOUNT|LONG_ERROR_RUN
- frame 240: HEAVIEST_UNDERCOUNT
- frame 241: HEAVIEST_UNDERCOUNT
- frame 242: HEAVIEST_UNDERCOUNT|LONG_ERROR_RUN
- frame 243: HEAVIEST_UNDERCOUNT
- frame 244: HEAVIEST_UNDERCOUNT
- frame 245: HEAVIEST_UNDERCOUNT
- frame 246: HEAVIEST_UNDERCOUNT|LONG_E

## 8. Extract annotated review images and initialize manual labels

`suspected_failure_mode` defaults to `UNREVIEWED`. USER review may later assign: `OCCLUSION`, `FISH_OVERLAP`, `WALL_EDGE`, `FRAME_EDGE`, `OBSTACLE`, `MOTION_BLUR`, `REFLECTION`, `LOW_CONTRAST`, `MERGED_BOX`, `DUPLICATE_BOX`, `LOW_CONFIDENCE`, `OTHER`, or `UNCLEAR`.

In [8]:
REVIEW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
video_capture = cv2.VideoCapture(str(VIDEO_PATH))
if not video_capture.isOpened(): raise RuntimeError('FAIL: cannot open source video for review extraction.')
review_rows = []
try:
    for index, frame_index in enumerate(REVIEW_FRAMES, 1):
        video_capture.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
        ok, frame = video_capture.read()
        if not ok or frame is None: raise RuntimeError(f'FAIL: cannot decode review frame {frame_index}')
        frame_record = FRAME_AUDIT.loc[FRAME_AUDIT['frame_index'] == frame_index].iloc[0]
        frame_detections = DETECTIONS[DETECTIONS['frame_index'] == frame_index]
        for detection in frame_detections.itertuples(index=False):
            p1, p2 = (int(round(detection.x1)), int(round(detection.y1))), (int(round(detection.x2)), int(round(detection.y2)))
            cv2.rectangle(frame, p1, p2, (0, 220, 0), 2)
            cv2.putText(frame, f'Ca {detection.confidence:.2f}', (p1[0], max(18, p1[1] - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 220, 0), 2, cv2.LINE_AA)
        header = f'frame={frame_index} t={frame_record.time_sec:.2f}s detected={int(frame_record.n_detections)} expected={EXPECTED_FISH_COUNT} {frame_record.error_type}'
        cv2.rectangle(frame, (0, 0), (min(VIDEO_WIDTH, 1000), 38), (0, 0, 0), -1)
        cv2.putText(frame, header, (10, 27), cv2.FONT_HERSHEY_SIMPLEX, 0.68, (255, 255, 255), 2, cv2.LINE_AA)
        error_tag = frame_record.error_type.lower(); filename = f'frame_{frame_index:06d}_{error_tag}_{int(frame_record.n_detections)}_vs_{EXPECTED_FISH_COUNT}.jpg'
        review_image_path = REVIEW_OUTPUT_DIR / filename
        if not cv2.imwrite(str(review_image_path), frame): raise RuntimeError(f'FAIL: cannot write {review_image_path}')
        review_rows.append({'frame_index': frame_index, 'time_sec': float(frame_record.time_sec), 'detected_count': int(frame_record.n_detections), 'expected_count': EXPECTED_FISH_COUNT, 'count_error': int(frame_record.count_error), 'error_type': frame_record.error_type, 'review_priority': '|'.join(sorted(review_reasons[frame_index])), 'review_image_path': str(review_image_path.relative_to(PROJECT_ROOT)), 'suspected_failure_mode': 'UNREVIEWED', 'review_note': ''})
        if index % 10 == 0 or index == len(REVIEW_FRAMES): print(f'Extracted review images: {index}/{len(REVIEW_FRAMES)}')
finally:
    video_capture.release()
FAILURE_REVIEW_DF = pd.DataFrame(review_rows)
assert len(FAILURE_REVIEW_DF) == len(REVIEW_FRAMES), 'FAIL: review image extraction incomplete.'
print(f'Review images created: {len(FAILURE_REVIEW_DF)} in {REVIEW_OUTPUT_DIR.relative_to(PROJECT_ROOT)}')

Extracted review images: 10/58
Extracted review images: 20/58
Extracted review images: 30/58
Extracted review images: 40/58
Extracted review images: 50/58
Extracted review images: 58/58
Review images created: 58 in outputs/front/detection/failure_audit/conf068_n1


## 9. Save evidence and cautious scientific interpretation

In [9]:
LOG_DIR = PROJECT_ROOT / 'logs' / 'detection' / EXPERIMENT_ID
RESULTS_DIR = PROJECT_ROOT / 'results' / 'detection'
LOG_DIR.mkdir(parents=True, exist_ok=True); RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = LOG_DIR / 'config.yaml'; ENVIRONMENT_PATH = LOG_DIR / 'environment.txt'; SUMMARY_PATH = LOG_DIR / 'summary.json'
FAILURE_SUMMARY_PATH = RESULTS_DIR / f'front_detection_failure_summary_{RUN_TAG}.csv'
ERROR_RUNS_PATH = RESULTS_DIR / f'front_detection_error_runs_{RUN_TAG}.csv'
FAILURE_REVIEW_PATH = RESULTS_DIR / f'front_detection_failure_review_{RUN_TAG}.csv'
EXACT_VS_ERROR_PATH = RESULTS_DIR / f'front_detection_exact_vs_error_{RUN_TAG}.csv'
FRAME_AUDIT_PATH = REVIEW_OUTPUT_DIR / 'frame_audit.csv'
CONFIG_EVIDENCE = dict(CONFIG, model_sha256=MODEL_SHA256, video=str(VIDEO_PATH.relative_to(PROJECT_ROOT)), video_sha256=VIDEO_SHA256, source_total_frames=TOTAL_FRAMES, source_fps=VIDEO_FPS, git_commit=GIT_COMMIT)
CONFIG_PATH.write_text(yaml.safe_dump(CONFIG_EVIDENCE, sort_keys=False), encoding='utf-8')
ENVIRONMENT_LINES = [f'experiment_id={EXPERIMENT_ID}', f'datetime_utc={STARTED_AT}', f'git_commit={GIT_COMMIT}', f'python_executable={sys.executable}', f'python={platform.python_version()}', f'conda_env={CONDA_ENV}', f'torch={torch.__version__}', f'cuda_available={CUDA_AVAILABLE}', f'gpu={GPU_NAME}', f'ultralytics={ultralytics.__version__}', f'opencv={cv2.__version__}']
ENVIRONMENT_PATH.write_text('\n'.join(ENVIRONMENT_LINES) + '\n', encoding='utf-8')
FRAME_AUDIT.to_csv(FRAME_AUDIT_PATH, index=False)
ERROR_RUNS_DF.to_csv(ERROR_RUNS_PATH, index=False); FAILURE_REVIEW_DF.to_csv(FAILURE_REVIEW_PATH, index=False); EXACT_VS_ERROR_DF.to_csv(EXACT_VS_ERROR_PATH, index=False)
FAILURE_SUMMARY_ROW = {'experiment_id': EXPERIMENT_ID, 'source_experiment_id': SOURCE_EXPERIMENT_ID, 'detection_conf': DETECTION_CONF, 'expected_fish_count': EXPECTED_FISH_COUNT, 'experimental_fish_count': EXPECTED_FISH_COUNT, 'total_frames': TOTAL_FRAMES, 'exact_frames': EXACT_FRAMES, 'exact_rate': EXACT_RATE, 'undercount_frames': UNDERCOUNT_FRAMES, 'undercount_rate': UNDERCOUNT_RATE, 'overcount_frames': OVERCOUNT_FRAMES, 'overcount_rate': OVERCOUNT_RATE, 'error_abs_1_frames': ERROR_ABS_1_FRAMES, 'error_abs_2_frames': ERROR_ABS_2_FRAMES, 'error_abs_ge3_frames': ERROR_ABS_GE3_FRAMES, 'error_runs': ERROR_RUN_COUNT, 'longest_error_run_frames': LONGEST_ERROR_RUN_FRAMES, 'longest_error_run_sec': LONGEST_ERROR_RUN_SEC, 'near_threshold_detections': NEAR_THRESHOLD_DETECTIONS, 'near_threshold_rate': NEAR_THRESHOLD_RATE, 'near_threshold_frames': NEAR_THRESHOLD_FRAMES, 'review_frames_created': len(FAILURE_REVIEW_DF)}
for row in SEVERITY_DF.itertuples(index=False): FAILURE_SUMMARY_ROW[f'{row.error_type.lower()}_{row.severity}_frames'] = row.frames
pd.DataFrame([FAILURE_SUMMARY_ROW]).to_csv(FAILURE_SUMMARY_PATH, index=False)
dominant_error = 'UNDERCOUNT' if UNDERCOUNT_FRAMES > OVERCOUNT_FRAMES else ('OVERCOUNT' if OVERCOUNT_FRAMES > UNDERCOUNT_FRAMES else 'BALANCED_OR_NONE')
run_pattern = 'temporally persistent' if LONGEST_ERROR_RUN_SEC >= 1.0 else 'mostly short-lived'
comparison_lookup = EXACT_VS_ERROR_DF.set_index('frame_group') if len(EXACT_VS_ERROR_DF) else pd.DataFrame()
confidence_association = 'cannot be compared because one group is absent'
if {'EXACT', 'ERROR'}.issubset(comparison_lookup.index):
    exact_conf, error_conf = comparison_lookup.loc['EXACT', 'mean_mean_detection_confidence'], comparison_lookup.loc['ERROR', 'mean_mean_detection_confidence']
    confidence_association = 'lower in ERROR frames' if error_conf < exact_conf else 'not lower in ERROR frames'
edge_rate = float((FRAME_AUDIT['near_edge_detections'] > 0).mean()); overlap_rate = float(FRAME_AUDIT['high_bbox_overlap'].mean())
INTERPRETATION = f'On the continuous raw video, count mismatch is dominated by {dominant_error.lower()} frames. Error runs appear {run_pattern}; the longest lasts {LONGEST_ERROR_RUN_SEC:.3f} s. Mean retained confidence is {confidence_association}. Near-edge diagnostics occur in {edge_rate:.3f} of frames and high-overlap diagnostics in {overlap_rate:.3f}; these are associations for manual review, not confirmed causes, FP/FN labels, behavior conclusions, or tracking conclusions.'
EVIDENCE_WARNINGS = ['Manual failure modes remain UNREVIEWED; automatic diagnostics are not ground-truth failure labels.', 'Previous N8 count diagnostics for this video were generated with an incorrect expected fish count and are superseded by the N1 run.']
if EXACT_FRAMES == 0: EVIDENCE_WARNINGS.append('No EXACT-count control frames were available at this threshold.')
CHECKPOINT_RESULT = 'PASS_WITH_WARNING'
OUTPUT_FILES = [CONFIG_PATH, ENVIRONMENT_PATH, SUMMARY_PATH, FAILURE_SUMMARY_PATH, ERROR_RUNS_PATH, FAILURE_REVIEW_PATH, EXACT_VS_ERROR_PATH]
SUMMARY = {**FAILURE_SUMMARY_ROW, 'model_sha256': MODEL_SHA256, 'video': str(VIDEO_PATH.relative_to(PROJECT_ROOT)), 'video_sha256': VIDEO_SHA256, 'interpretation': INTERPRETATION, 'checkpoint_result': CHECKPOINT_RESULT, 'warnings': EVIDENCE_WARNINGS, 'output_files': [str(path.relative_to(PROJECT_ROOT)) for path in OUTPUT_FILES], 'local_frame_audit': str(FRAME_AUDIT_PATH.relative_to(PROJECT_ROOT)), 'review_image_directory': str(REVIEW_OUTPUT_DIR.relative_to(PROJECT_ROOT)), 'git_commit': GIT_COMMIT, 'next_step': 'USER manually reviews failure images and labels before any Notebook 07 work.'}
SUMMARY_PATH.write_text(json.dumps(SUMMARY, indent=2, ensure_ascii=False), encoding='utf-8')
print('SCIENTIFIC INTERPRETATION')
print(INTERPRETATION)
for path in OUTPUT_FILES: print(f'Created {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size} bytes)')

SCIENTIFIC INTERPRETATION
On the continuous raw video, count mismatch is dominated by undercount frames. Error runs appear temporally persistent; the longest lasts 36.626 s. Mean retained confidence is not lower in ERROR frames. Near-edge diagnostics occur in 0.000 of frames and high-overlap diagnostics in 0.000; these are associations for manual review, not confirmed causes, FP/FN labels, behavior conclusions, or tracking conclusions.
Created logs/detection/FRONT_DET_FAILURE_AUDIT_CONF068_N1_001/config.yaml (889 bytes)
Created logs/detection/FRONT_DET_FAILURE_AUDIT_CONF068_N1_001/environment.txt (346 bytes)
Created logs/detection/FRONT_DET_FAILURE_AUDIT_CONF068_N1_001/summary.json (2951 bytes)
Created results/detection/front_detection_failure_summary_conf068_n1.csv (933 bytes)
Created results/detection/front_detection_error_runs_conf068_n1.csv (1701 bytes)
Created results/detection/front_detection_failure_review_conf068_n1.csv (8968 bytes)
Created results/detection/front_detection_exa

## 10. Final Summary

In [10]:
FINAL_SUMMARY = {'experiment_id': EXPERIMENT_ID, 'source_experiment_id': SOURCE_EXPERIMENT_ID, 'detection_conf': DETECTION_CONF, 'expected_fish_count': EXPECTED_FISH_COUNT, 'experimental_fish_count': EXPECTED_FISH_COUNT, 'total_frames': TOTAL_FRAMES, 'exact_frames': EXACT_FRAMES, 'exact_rate': EXACT_RATE, 'undercount_frames': UNDERCOUNT_FRAMES, 'undercount_rate': UNDERCOUNT_RATE, 'overcount_frames': OVERCOUNT_FRAMES, 'overcount_rate': OVERCOUNT_RATE, 'error_abs_1_frames': ERROR_ABS_1_FRAMES, 'error_abs_2_frames': ERROR_ABS_2_FRAMES, 'error_abs_ge3_frames': ERROR_ABS_GE3_FRAMES, 'error_runs': ERROR_RUN_COUNT, 'longest_error_run_frames': LONGEST_ERROR_RUN_FRAMES, 'longest_error_run_sec': LONGEST_ERROR_RUN_SEC, 'near_threshold_detections': NEAR_THRESHOLD_DETECTIONS, 'near_threshold_rate': NEAR_THRESHOLD_RATE, 'review_frames_created': len(FAILURE_REVIEW_DF), 'checkpoint_result': CHECKPOINT_RESULT, 'warnings': EVIDENCE_WARNINGS, 'output_files': [str(path.relative_to(PROJECT_ROOT)) for path in OUTPUT_FILES], 'next_step': SUMMARY['next_step']}
print('FINAL SUMMARY')
for key, value in FINAL_SUMMARY.items(): print(f'{key}: {value}')

FINAL SUMMARY
experiment_id: FRONT_DET_FAILURE_AUDIT_CONF068_N1_001
source_experiment_id: FRONT_VIDEO_DET_CONF068_N1_001
detection_conf: 0.68
expected_fish_count: 1
experimental_fish_count: 1
total_frames: 3431
exact_frames: 2301
exact_rate: 0.6706499562809677
undercount_frames: 1130
undercount_rate: 0.32935004371903237
overcount_frames: 0
overcount_rate: 0.0
error_abs_1_frames: 1130
error_abs_2_frames: 0
error_abs_ge3_frames: 0
error_runs: 32
longest_error_run_frames: 1050
longest_error_run_sec: 36.62565155931215
near_threshold_detections: 1663
near_threshold_rate: 0.722729248152977
review_frames_created: 58
checkpoint_result: PASS_WITH_WARNING
warnings: ['Manual failure modes remain UNREVIEWED; automatic diagnostics are not ground-truth failure labels.', 'Previous N8 count diagnostics for this video were generated with an incorrect expected fish count and are superseded by the N1 run.']
output_files: ['logs/detection/FRONT_DET_FAILURE_AUDIT_CONF068_N1_001/config.yaml', 'logs/detectio